<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</center>
</figure>

# **Universidad de Buenos Aires**
## **Facultad de Ciencias Económicas**

### **Taller de Programación para el Análisis de Datos**

# 📊 Estadística Descriptiva con Python
### Taller de Programación

---

En esta clase estudiamos las principales **medidas estadísticas descriptivas**:

| Grupo | Qué miden | Ejemplos |
|---|---|---|
| **Posición** | Dónde se ubican los datos | Media, Mediana, Moda, Percentiles |
| **Dispersión** | Qué tan esparcidos están | Varianza, Desvío estándar, RIQ |
| **Asimetría** | Qué tan simétrica es la distribución | Coef. de Pearson, Fisher (g1) |

**Estructura de la clase:**
- 🐍 **Parte 1** — Python puro: implementaciones a mano + módulo `statistics`
- ⚡ **Parte 2** — NumPy: operaciones vectorizadas
- 🐼 **Parte 3** — Pandas: análisis sobre DataFrames

---
# 🐍 PARTE 1 — Python Puro

Esta parte tiene **dos secciones**:
1. Implementación **desde cero** de cada medida (para entender las fórmulas)
2. Las mismas medidas usando el módulo `statistics` de la librería estándar de Python

### 📦 Dataset de trabajo

Vamos a trabajar con las notas de un curso (sobre 10 puntos).

In [5]:
notas = [4, 6, 7, 5, 8, 9, 6, 7, 3, 8, 7, 6, 9, 5, 7, 8, 4, 6, 10, 7]

print(f"Dataset: {notas}")
print(f"n = {len(notas)}")

Dataset: [4, 6, 7, 5, 8, 9, 6, 7, 3, 8, 7, 6, 9, 5, 7, 8, 4, 6, 10, 7]
n = 20


---
## 🔧 1.1 — Implementaciones a mano

Implementamos cada medida **desde cero**, traduciendo directamente las fórmulas matemáticas a código.

### Medidas de posición

In [6]:
# ── Media aritmética ─────────────────────────────────────────────
# x̄ = (1/n) * Σ xi

def media(datos):
    resultado = sum(datos) / len(datos)
    return resultado


# ── Mediana ──────────────────────────────────────────────────────
# Valor central de los datos ordenados.
# n impar  → elemento del medio
# n par    → promedio de los dos elementos centrales

def mediana(datos):
    ordenados = sorted(datos)
    n = len(ordenados)
    mitad = n // 2
    
    if n % 2 == 1:
        resultado_mediana=ordenados[mitad]
    else:
        resultado_mediana= (ordenados[mitad - 1] + ordenados[mitad]) / 2
    
    return resultado_mediana


# ── Moda ─────────────────────────────────────────────────────────
# El o los valores que más se repiten. (variables discretas)

def moda(datos):
    frecuencias = {}
    for x in datos:
        frecuencias[x] = frecuencias.get(x, 0) + 1
    max_freq = max(frecuencias.values())
    modas = sorted([v for v, f in frecuencias.items() if f == max_freq])
    return modas, max_freq


# ── Percentil (con interpolación lineal) ─────────────────────────
# Equivalente al método por defecto de NumPy.

def percentil(datos, p):
    ordenados = sorted(datos)
    n = len(ordenados)
    indice = (p / 100) * (n - 1)
    inf = int(indice)
    frac = indice - inf
    if inf + 1 < n:
        return ordenados[inf] + frac * (ordenados[inf + 1] - ordenados[inf])
    resultado=ordenados[inf]
    
    return resultado


mod, freq = moda(notas)
print("── POSICIÓN (a mano) ──")
print(f"Media   : {media(notas):.4f}")
print(f"Mediana : {mediana(notas)}")
print(f"Moda    : {mod}  (frec. {freq})")
print(f"Q1      : {percentil(notas, 25)}")
print(f"Q2      : {percentil(notas, 50)}")
print(f"Q3      : {percentil(notas, 75)}")

── POSICIÓN (a mano) ──
Media   : 6.6000
Mediana : 7.0
Moda    : [7]  (frec. 5)
Q1      : 5.75
Q2      : 7.0
Q3      : 8.0


### Medidas de dispersión

In [7]:
# ── Varianza ─────────────────────────────────────────────────────
# Poblacional: divide por N
# Muestral:   divide por N-1  (corrección de Bessel → estimador insesgado)

def varianza(datos, poblacional=False):
    m = media(datos)
    n = len(datos)
    divisor = n if poblacional else n - 1
    resultado=sum((x - m) ** 2 for x in datos) / divisor
    
    return resultado

def desvio_estandar(datos, poblacional=False):
    resultado=varianza(datos, poblacional) ** 0.5
    return resultado

def rango(datos):
    resultado=max(datos) - min(datos)
    return resultado

def riq(datos):
    resultado=percentil(datos, 75) - percentil(datos, 25)
    return resultado

def coef_variacion(datos):
    """Desvío estándar expresado como % de la media. Adimensional."""
    resultado=(desvio_estandar(datos) / media(datos)) * 100
    return resultado

def detectar_outliers(datos):
    """Regla de Tukey: outlier si x < Q1 - 1.5*RIQ  o  x > Q3 + 1.5*RIQ"""
    q1, q3 = percentil(datos, 25), percentil(datos, 75)
    r = riq(datos)
    lim_inf, lim_sup = q1 - 1.5 * r, q3 + 1.5 * r
    resultado=[x for x in datos if x < lim_inf or x > lim_sup], lim_inf, lim_sup
    return resultado


print("── DISPERSIÓN (a mano) ──")
print(f"Rango            : {rango(notas)}")
print(f"RIQ              : {riq(notas)}")
print(f"Varianza muestral: {varianza(notas):.4f}")
print(f"Desvío estándar  : {desvio_estandar(notas):.4f}")
print(f"CV               : {coef_variacion(notas):.2f}%")

notas_con_outlier = notas + [1]
outliers, li, ls = detectar_outliers(notas_con_outlier)
print(f"\nOutliers (Tukey): {outliers}  →  límites [{li:.2f}, {ls:.2f}]")

── DISPERSIÓN (a mano) ──
Rango            : 7
RIQ              : 2.25
Varianza muestral: 3.3053
Desvío estándar  : 1.8180
CV               : 27.55%

Outliers (Tukey): []  →  límites [0.50, 12.50]


---
## 📚 1.2 — Módulo `statistics` (librería estándar)

Python incluye el módulo `statistics` sin necesidad de instalar nada. Cubre la mayoría de las medidas vistas.

> ⚠️ **Lo que NO tiene:** rango, RIQ, coeficiente de variación, detección de outliers, asimetría → para eso seguimos necesitando la implementación propia.

In [8]:
import statistics

print("── POSICIÓN (statistics) ──")
print(f"Media          : {statistics.mean(notas):.4f}")
print(f"Mediana        : {statistics.median(notas)}")
print(f"Moda           : {statistics.mode(notas)}")
print(f"Multimode      : {statistics.multimode(notas)}   ← devuelve TODAS las modas")

# quantiles divide los datos en n partes iguales
# n=4 → cuartiles, n=10 → deciles, n=100 → percentiles
cuartiles = statistics.quantiles(notas, n=4)
print(f"\nCuartiles [Q1, Q2, Q3] : {cuartiles}")

deciles = statistics.quantiles(notas, n=10)
print(f"Deciles (D1..D9)       : {[round(d, 2) for d in deciles]}")

── POSICIÓN (statistics) ──
Media          : 6.6000
Mediana        : 7.0
Moda           : 7
Multimode      : [7]   ← devuelve TODAS las modas

Cuartiles [Q1, Q2, Q3] : [5.25, 7.0, 8.0]
Deciles (D1..D9)       : [4.0, 5.0, 6.0, 6.0, 7.0, 7.0, 7.7, 8.0, 9.0]


In [9]:
print("── DISPERSIÓN (statistics) ──")
print(f"Varianza muestral    : {statistics.variance(notas):.4f}   ← ddof=1")
print(f"Varianza poblacional : {statistics.pvariance(notas):.4f}  ← ddof=0")
print(f"Desvío muestral      : {statistics.stdev(notas):.4f}")
print(f"Desvío poblacional   : {statistics.pstdev(notas):.4f}")

# Rango e IQR no están en statistics → usamos las funciones propias
print(f"\nRango (impl. propia) : {rango(notas)}")
print(f"RIQ   (impl. propia) : {riq(notas)}")

── DISPERSIÓN (statistics) ──
Varianza muestral    : 3.3053   ← ddof=1
Varianza poblacional : 3.1400  ← ddof=0
Desvío muestral      : 1.8180
Desvío poblacional   : 1.7720

Rango (impl. propia) : 7
RIQ   (impl. propia) : 2.25


### Comparación: implementación propia vs `statistics`

In [10]:
print(f"{'Medida':<22} {'A mano':>12} {'statistics':>12} {'¿Coinciden?':>12}")
print("─" * 60)

comparaciones = [
    ("Media",             media(notas),            statistics.mean(notas)),
    ("Mediana",           mediana(notas),          statistics.median(notas)),
    ("Varianza muestral", varianza(notas),         statistics.variance(notas)),
    ("Desvío muestral",   desvio_estandar(notas),  statistics.stdev(notas)),
]

for nombre, val_manual, val_stats in comparaciones:
    ok = "✅" if abs(val_manual - val_stats) < 1e-9 else "❌"
    print(f"{nombre:<22} {val_manual:>12.6f} {val_stats:>12.6f} {ok:>12}")

Medida                       A mano   statistics  ¿Coinciden?
────────────────────────────────────────────────────────────
Media                      6.600000     6.600000            ✅
Mediana                    7.000000     7.000000            ✅
Varianza muestral          3.305263     3.305263            ✅
Desvío muestral            1.818038     1.818038            ✅


---
## 📐 1.3 — Asimetría (Skewness)

La asimetría mide **qué tan simétrica es la distribución** de los datos respecto a su centro.

```
Asimetría negativa        Simétrica         Asimetría positiva
(cola a la izquierda)    (sin cola)         (cola a la derecha)

     ▐█▌                   ▐█▌                    ▐█▌
    ▐███▌                 ▐███▌                  ▐███▌
   ▐█████▌               ▐█████▌                ▐█████▌
 ▐█████████▌           ▐█████████▌            ▐█████████▌
▐▬▬▬▬▬▬▬▬▬▬▬▬▌        ▐▬▬▬▬▬▬▬▬▬▬▌          ▐▬▬▬▬▬▬▬▬▬▬▬▬▌
media < mediana       media ≈ mediana        media > mediana
```

> ⚠️ El módulo `statistics` **no incluye** funciones de asimetría → implementamos a mano y luego usamos scipy/pandas.

### Coeficiente de Pearson

Dos variantes, basadas en la diferencia entre la media y otra medida central:

$$AS = \frac{3(\bar{x} - Me)}{s} \qquad \text{( coeficiente: usa la mediana, más estable)}$$

**Interpretación:**
- $AS > 0$ → cola a la derecha (media > mediana)
- $AS < 0$ → cola a la izquierda (media < mediana)
- $AS \approx 0$ → distribución aproximadamente simétrica

In [11]:
def asimetria_pearson(datos):
    """2do coeficiente de Pearson: 3*(media - mediana) / desvío"""
    resultado=3 * (media(datos) - mediana(datos)) / desvio_estandar(datos)
    return resultado


print(f"Media   = {media(notas):.4f}")
print(f"Mediana = {mediana(notas)}")
print(f"Moda    = {moda(notas)[0]}")
print(f"Coef. Pearson (mediana): {asimetria_pearson(notas):.4f}")

Media   = 6.6000
Mediana = 7.0
Moda    = [7]
Coef. Pearson (mediana): -0.6601


### Coeficiente de Fisher (g1) — momento de orden 3

Es el coeficiente **más utilizado** en estadística y el que implementan NumPy/SciPy, Pandas y la mayoría de los softwares.
Se basa en el **tercer momento estandarizado**:

$$g_1 = \frac{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^3}{s^3} \qquad \text{(poblacional)}$$

La versión **ajustada** para muestras (la que usa scipy y pandas) aplica un factor de corrección:

$$G_1 = \frac{n}{(n-1)(n-2)} \cdot \frac{\sum(x_i - \bar{x})^3}{s^3}$$

> Misma lógica que la corrección de Bessel: la muestra tiende a subestimar la asimetría poblacional.

In [37]:
def asimetria_fisher(datos, ajustado=True):
    """
    Coeficiente de asimetría de Fisher (g1 / G1).

    ajustado=True  → versión muestral corregida (equivalente a scipy/pandas)
    ajustado=False → tercer momento estandarizado sin corrección
    """
    n = len(datos)
    m = media(datos)
    s = desvio_estandar(datos, poblacional=False)

    tercer_momento = sum((x - m) ** 3 for x in datos) / n
    g1 = tercer_momento / s ** 3

    if ajustado:
        factor = (n * (n - 1)) ** 0.5 / (n - 2)
        return factor * g1
    return g1


print(f"Fisher G1 (ajustado):    {asimetria_fisher(notas, ajustado=True):.4f}  ← equivalente a scipy/pandas")

valor = asimetria_fisher(notas)
if abs(valor) < 0.5:
    desc = "aproximadamente simétrica"
elif valor > 0:
    desc = "asimetría positiva (cola derecha)"
else:
    desc = "asimetría negativa (cola izquierda)"
print(f"\nInterpretación: G1 = {valor:.4f}  →  {desc}")

Fisher G1 (ajustado):    -0.1384  ← equivalente a scipy/pandas

Interpretación: G1 = -0.1384  →  aproximadamente simétrica


### Comparación con datasets de distinta forma

In [13]:
# Cola a la derecha: pocos ingresos muy altos "estiran" hacia arriba
ingresos = [15, 18, 20, 22, 22, 23, 24, 25, 26, 27,
            28, 28, 29, 30, 31, 32, 35, 40, 55, 120]

# Cola a la izquierda: pocos tiempos muy bajos
tiempos  = [1, 8, 12, 15, 17, 18, 18, 19, 19, 20,
            20, 20, 21, 21, 22, 22, 23, 23, 24, 25]

print(f"{'Dataset':<12} {'Media':>8} {'Mediana':>8} {'G1':>10} {'Tipo':>28}")
print("─" * 72)

for nombre, datos in [("Notas", notas), ("Ingresos", ingresos), ("Tiempos", tiempos)]:
    m   = media(datos)
    med = mediana(datos)
    g   = asimetria_fisher(datos)
    tipo = "positiva (cola →)" if g > 0.5 else "negativa (← cola)" if g < -0.5 else "aprox. simétrica"
    print(f"{nombre:<12} {m:>8.2f} {med:>8.2f} {g:>10.4f} {tipo:>28}")

Dataset         Media  Mediana         G1                         Tipo
────────────────────────────────────────────────────────────────────────
Notas            6.60     7.00    -0.1384             aprox. simétrica
Ingresos        32.50    27.50     3.2428            positiva (cola →)
Tiempos         18.40    20.00    -1.6696            negativa (← cola)


### Resumen completo — Python puro

In [15]:
def resumen_estadistico(datos, nombre="Dataset"):
    """Resumen estadístico completo: posición, dispersión y asimetría."""
    n    = len(datos)
    m    = media(datos)
    med  = mediana(datos)
    mod, fmod = moda(datos)
    q1   = percentil(datos, 25)
    q3   = percentil(datos, 75)
    s    = desvio_estandar(datos)
    g    = asimetria_fisher(datos)
    ap2  = asimetria_pearson(datos)

    print(f"{'═'*44}")
    print(f"  {nombre}  (n={n})")
    print(f"{'─'*44}")
    print(f"  POSICIÓN")
    print(f"  Media            : {m:.4f}")
    print(f"  Mediana          : {med}")
    print(f"  Moda             : {mod}  (frec. {fmod})")
    print(f"  Q1 / Q3          : {q1} / {q3}")
    print(f"{'─'*44}")
    print(f"  DISPERSIÓN")
    print(f"  Mín / Máx        : {min(datos)} / {max(datos)}")
    print(f"  Rango            : {rango(datos)}")
    print(f"  RIQ              : {riq(datos)}")
    print(f"  Varianza         : {varianza(datos):.4f}")
    print(f"  Desvío estándar  : {s:.4f}")
    print(f"  CV               : {coef_variacion(datos):.2f}%")
    print(f"{'─'*44}")
    print(f"  ASIMETRÍA")
    print(f"  Pearson 2        : {ap2:.4f}")
    print(f"  Fisher G1        : {g:.4f}")
    tipo = "positiva (cola →)" if g > 0.5 else "negativa (← cola)" if g < -0.5 else "~ simétrica"
    print(f"  Tipo             : {tipo}")
    print(f"{'═'*44}")

resumen_estadistico(notas, "Notas del curso")

════════════════════════════════════════════
  Notas del curso  (n=20)
────────────────────────────────────────────
  POSICIÓN
  Media            : 6.6000
  Mediana          : 7.0
  Moda             : [7]  (frec. 5)
  Q1 / Q3          : 5.75 / 8.0
────────────────────────────────────────────
  DISPERSIÓN
  Mín / Máx        : 3 / 10
  Rango            : 7
  RIQ              : 2.25
  Varianza         : 3.3053
  Desvío estándar  : 1.8180
  CV               : 27.55%
────────────────────────────────────────────
  ASIMETRÍA
  Pearson 2        : -0.6601
  Fisher G1        : -0.1384
  Tipo             : ~ simétrica
════════════════════════════════════════════


---
## ✏️ Ejercicios — Parte 1

**Ejercicio 1.** Dado el siguiente dataset de tiempos de respuesta de una API (en ms), producí un resumen estadístico completo e identificá si hay outliers.

In [ ]:
tiempos_ms = [120, 135, 128, 142, 119, 310, 127, 133, 125, 140,
              122, 138, 131, 129, 136, 124, 141, 126, 132, 118]

# Tu código aquí


**Ejercicio 2.** ¿Por qué el coeficiente de Fisher de `ingresos` es mucho mayor que el de `notas`? Verificalo mirando la diferencia entre media y mediana.

**Ejercicio 3.** Usando `statistics.quantiles()`, calculá los deciles de `tiempos_ms`. ¿Qué porcentaje de los tiempos está por debajo de 135ms?

**Ejercicio 4.** ¿Cómo cambia la asimetría de `tiempos_ms` si eliminás el outlier 310? ¿Se vuelve más o menos simétrico?

In [ ]:
# Ejercicios 2, 3 y 4


---
# ⚡ PARTE 2 — NumPy

Las mismas operaciones, vectorizadas y optimizadas. Para asimetría usamos `scipy.stats`.

In [16]:
import numpy as np
from scipy import stats

arr = np.array(notas)
print(f"Array: {arr}  |  dtype: {arr.dtype}")

C:\Users\cmarc\AppData\Local\Temp\ipykernel_12440\2303819103.py:2: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.2)
  from scipy import stats


Array: [ 4  6  7  5  8  9  6  7  3  8  7  6  9  5  7  8  4  6 10  7]  |  dtype: int64


In [17]:
print("── POSICIÓN ──")
print(f"Media          : {np.mean(arr):.4f}")
print(f"Mediana        : {np.median(arr)}")
vals, cnts = np.unique(arr, return_counts=True)
print(f"Moda           : {vals[np.argmax(cnts)]}  (frec. {np.max(cnts)})")
q1, q2, q3 = np.percentile(arr, [25, 50, 75])
print(f"Q1 / Q2 / Q3   : {q1} / {q2} / {q3}")
print(f"P90            : {np.percentile(arr, 90)}")

── POSICIÓN ──
Media          : 6.6000
Mediana        : 7.0
Moda           : 7  (frec. 5)
Q1 / Q2 / Q3   : 5.75 / 7.0 / 8.0
P90            : 9.0


In [18]:
# ⚠️ ddof=0 → poblacional (default NumPy)
#    ddof=1 → muestral (para comparar con statistics/pandas)

print("── DISPERSIÓN ──")
print(f"Rango          : {np.ptp(arr)}")
print(f"RIQ            : {np.percentile(arr, 75) - np.percentile(arr, 25)}")
print(f"Varianza muestral   : {np.var(arr, ddof=1):.4f}")
print(f"Desvío muestral     : {np.std(arr, ddof=1):.4f}")
print(f"CV             : {np.std(arr, ddof=1) / np.mean(arr) * 100:.2f}%")

── DISPERSIÓN ──
Rango          : 7
RIQ            : 2.25
Varianza muestral   : 3.3053
Desvío muestral     : 1.8180
CV             : 27.55%


In [19]:
# scipy.stats.skew:
#   bias=False → versión ajustada G1 (equivalente a pandas)
#   bias=True  → sin ajuste, g1

print("── ASIMETRÍA ──")
G1_scipy      = stats.skew(arr, bias=False)
g1_sin_ajuste = stats.skew(arr, bias=True)

print(f"Fisher G1 (bias=False, ajustado): {G1_scipy:.4f}")
print(f"Fisher g1 (bias=True, sin ajuste): {g1_sin_ajuste:.4f}")
print(f"Nuestra impl. (ajustado):          {asimetria_fisher(notas):.4f}  ✅")

── ASIMETRÍA ──
Fisher G1 (bias=False, ajustado): -0.1495
Fisher g1 (bias=True, sin ajuste): -0.1380
Nuestra impl. (ajustado):          -0.1384  ✅


### Operaciones sobre matrices 2D — el eje `axis`

In [20]:
# Notas de 4 estudiantes en 5 TPs
matriz = np.array([
    [8, 7, 9, 6, 8],
    [5, 6, 5, 7, 6],
    [9, 10, 8, 9, 10],
    [4, 5, 6, 4, 5],
])

print("Promedio por estudiante (axis=1):", np.mean(matriz, axis=1))
print("Promedio por TP         (axis=0):", np.mean(matriz, axis=0))
print("Desvío por estudiante   (axis=1):", np.std(matriz, axis=1, ddof=1).round(2))
print("Asimetría por TP        (axis=0):", stats.skew(matriz, axis=0, bias=False).round(4))

Promedio por estudiante (axis=1): [7.6 5.8 9.2 4.8]
Promedio por TP         (axis=0): [6.5  7.   7.   6.5  7.25]
Desvío por estudiante   (axis=1): [1.14 0.84 0.84 0.84]
Asimetría por TP        (axis=0): [0.     1.1903 0.     0.     0.4816]


---
# 🐼 PARTE 3 — Pandas

In [21]:
import pandas as pd

data = {
    "estudiante" : [f"E{i:02d}" for i in range(1, 21)],
    "nota_tp"    : [4, 6, 7, 5, 8, 9, 6, 7, 3, 8, 7, 6, 9, 5, 7, 8, 4, 6, 10, 7],
    "nota_examen": [5, 7, 6, 4, 9, 8, 7, 8, 2, 7, 6, 5, 10, 6, 8, 7, 3, 5, 9, 6],
    "asistencia" : [80, 95, 70, 60, 100, 90, 85, 75, 50, 88,
                    72, 91, 98, 65, 83, 77, 55, 88, 100, 93],
}

df = pd.DataFrame(data)
df.head()

,estudiante,nota_tp,nota_examen,asistencia
0,E01,4,5,80
1,E02,6,7,95
2,E03,7,6,70
3,E04,5,4,60
4,E05,8,9,100


In [22]:
# describe() cubre posición y dispersión automáticamente
df.describe(percentiles=[0.25, 0.50, 0.75, 0.90])

,nota_tp,nota_examen,asistencia
count,20.000000,20.000000,20.000000
mean,6.600000,6.400000,80.750000
std,1.818038,2.036509,14.934506
min,3.000000,2.000000,50.000000
25%,5.750000,5.000000,71.500000
50%,7.000000,6.500000,84.000000
75%,8.000000,8.000000,91.500000
90%,9.000000,9.000000,98.200000
max,10.000000,10.000000,100.000000


In [23]:
cols = ["nota_tp", "nota_examen", "asistencia"]

# df.skew() usa G1 ajustado → equivalente a scipy bias=False
print("── ASIMETRÍA (Pandas) ──")
print(df[cols].skew().round(4))

print("\n── Tabla resumen completa ──")
resumen = pd.DataFrame({
    "media"     : df[cols].mean(),
    "mediana"   : df[cols].median(),
    "desvío"    : df[cols].std(),
    "CV (%)"    : (df[cols].std() / df[cols].mean() * 100).round(2),
    "RIQ"       : df[cols].quantile(0.75) - df[cols].quantile(0.25),
    "asimetría" : df[cols].skew(),
}).T

print(resumen.round(4))

── ASIMETRÍA (Pandas) ──
nota_tp       -0.1495
nota_examen   -0.3589
asistencia    -0.6021
dtype: float64

── Tabla resumen completa ──
           nota_tp  nota_examen  asistencia
media       6.6000       6.4000     80.7500
mediana     7.0000       6.5000     84.0000
desvío      1.8180       2.0365     14.9345
CV (%)     27.5500      31.8200     18.4900
RIQ         2.2500       3.0000     20.0000
asimetría  -0.1495      -0.3589     -0.6021


### Análisis por grupos con `groupby()`

In [34]:
df["grupo"] = pd.cut(
    df["asistencia"],
    bins=[0, 70, 85, 100],
    labels=["Baja", "Media", "Alta"]
)

resumen_grupo = df.groupby("grupo", observed=True)[["asistencia"]].count().round(3)

print(resumen_grupo)

       asistencia
grupo            
Baja            5
Media           6
Alta            9


---
## ✏️ Ejercicios — Parte 2 y 3

**Ejercicio 5.** Calculá los percentiles [10, 25, 50, 75, 90] de `tiempos_ms` con NumPy. ¿Qué dice el P90 sobre el SLA del servicio?

**Ejercicio 6.** Calculá la asimetría de Fisher de `ingresos` y `tiempos` con `scipy.stats.skew`. ¿Coincide con nuestra implementación manual?

**Ejercicio 7.** Extendé el DataFrame con una columna `aprobado = nota_examen >= 6`. Con `groupby()`, calculá media, desvío y asimetría de asistencia para aprobados y desaprobados.

**Ejercicio 8.** Cargá el siguiente CSV, producí un resumen completo e identificá qué columna tiene mayor asimetría. ¿Qué significa eso en el contexto del negocio?

In [ ]:
import io

csv_data = """producto,precio,stock,ventas_mes
A,150,200,45
B,2500,30,8
C,80,500,120
D,1200,15,3
E,45,800,300
F,990,60,25
G,320,150,60
H,75,400,200
"""

df_ventas = pd.read_csv( io.StringIO(csv_data) )

# Tu análisis aquí


In [36]:
df_ventas.head()

,producto,precio,stock,ventas_mes
0,A,150,200,45
1,B,2500,30,8
2,C,80,500,120
3,D,1200,15,3
4,E,45,800,300
